# Export MIDI-DDSP Decoder to ONNX

This notebook exports the pretrained MIDI-DDSP synthesis decoder to ONNX format.
The exported model maps `(f0_hz, loudness_db, instrument_id)` → DDSP synthesis parameters.

**Instructions:** Runtime → Run all. Download the `.onnx` file when prompted.

In [ ]:
# 1. Install dependencies
!pip install -q midi-ddsp tf2onnx onnx

In [ ]:
# 2. Download pretrained MIDI-DDSP weights
from midi_ddsp.download_model_weights import main as download_weights
download_weights()
print('Weights downloaded.')

In [ ]:
# 3. Load the pretrained model
from midi_ddsp.utils.midi_synthesis_utils import load_pretrained_model

synthesis_generator, expression_generator = load_pretrained_model()
print(f'Loaded synthesis_generator: {type(synthesis_generator)}')
print(f'Loaded expression_generator: {type(expression_generator)}')

In [ ]:
# 4. Build a concrete TF function for the decoder path
import tensorflow as tf
import numpy as np

N_FRAMES = 250  # 1 second at 250 Hz

# Trace the synthesis path: (f0, loudness, instrument_id) → synth params
# We call the model's forward pass with example inputs to trace the graph.
example_f0 = tf.constant(np.full((1, N_FRAMES, 1), 440.0, dtype=np.float32))
example_ld = tf.constant(np.full((1, N_FRAMES, 1), -30.0, dtype=np.float32))
example_inst = tf.constant([[0]], dtype=tf.int32)  # violin

# Run once to trace
try:
    outputs = synthesis_generator(example_f0, example_ld, example_inst, training=False)
    print('Direct call works.')
    print('Output keys:', list(outputs.keys()) if isinstance(outputs, dict) else type(outputs))
except Exception as e:
    print(f'Direct call failed: {e}')
    print('Trying alternative call signature...')
    # Try different input formats
    try:
        outputs = synthesis_generator(
            {'f0_hz': example_f0[:,:,0], 'loudness_db': example_ld[:,:,0], 'instrument_id': example_inst[:,0]},
            training=False
        )
        print('Dict call works.')
        print('Output keys:', list(outputs.keys()) if isinstance(outputs, dict) else type(outputs))
    except Exception as e2:
        print(f'Dict call also failed: {e2}')

In [ ]:
# 5. Inspect the model to find the correct call signature
print('Model class:', synthesis_generator.__class__.__name__)
print('\nModel methods:')
for attr in dir(synthesis_generator):
    if not attr.startswith('_') and callable(getattr(synthesis_generator, attr, None)):
        print(f'  {attr}')

# Check if there's a call or predict method signature
import inspect
if hasattr(synthesis_generator, 'call'):
    print('\ncall signature:', inspect.signature(synthesis_generator.call))
if hasattr(synthesis_generator, 'forward'):
    print('\nforward signature:', inspect.signature(synthesis_generator.forward))
if hasattr(synthesis_generator, 'synthesize'):
    print('\nsynthesize signature:', inspect.signature(synthesis_generator.synthesize))

In [ ]:
# 6. Export to SavedModel
import os

SAVED_MODEL_DIR = '/tmp/ddsp_synthesis_generator'

# Get a concrete function that tf2onnx can convert
@tf.function(input_signature=[
    tf.TensorSpec([1, None], tf.float32, name='f0_hz'),
    tf.TensorSpec([1, None], tf.float32, name='loudness_db'),
    tf.TensorSpec([1], tf.int32, name='instrument_id'),
])
def export_fn(f0_hz, loudness_db, instrument_id):
    # Reshape to match model expectations
    seq_len = tf.shape(f0_hz)[1]
    f0 = tf.expand_dims(f0_hz, -1)  # [1, N, 1]
    ld = tf.expand_dims(loudness_db, -1)  # [1, N, 1]
    inst = tf.expand_dims(instrument_id, 0)  # [1, 1]
    
    # Call the synthesis generator
    outputs = synthesis_generator(f0, ld, inst, training=False)
    
    return {
        'amplitudes': outputs['amplitudes'],
        'harmonic_distribution': outputs['harmonic_distribution'],
        'noise_magnitudes': outputs['noise_magnitudes'],
    }

# Trace with example
cf = export_fn.get_concrete_function(
    tf.constant(np.full((1, N_FRAMES), 440.0, dtype=np.float32)),
    tf.constant(np.full((1, N_FRAMES), -30.0, dtype=np.float32)),
    tf.constant([0], dtype=np.int32),
)

tf.saved_model.save(synthesis_generator, SAVED_MODEL_DIR, signatures={'serving_default': cf})
print(f'SavedModel exported to {SAVED_MODEL_DIR}')

In [ ]:
# 7. Convert SavedModel to ONNX
import tf2onnx
import onnx

OUTPUT_PATH = '/tmp/ddsp_decoder.onnx'

model_proto, _ = tf2onnx.convert.from_saved_model(
    SAVED_MODEL_DIR,
    output_path=OUTPUT_PATH,
)

# Verify
model = onnx.load(OUTPUT_PATH)
onnx.checker.check_model(model)

file_size = os.path.getsize(OUTPUT_PATH)
print(f'\nONNX exported: {OUTPUT_PATH}')
print(f'Size: {file_size / 1024 / 1024:.1f} MB')
print(f'Inputs: {[inp.name for inp in model.graph.input]}')
print(f'Outputs: {[out.name for out in model.graph.output]}')

In [ ]:
# 8. Quick validation — run inference with ONNX Runtime
!pip install -q onnxruntime
import onnxruntime as ort

sess = ort.InferenceSession(OUTPUT_PATH)
print('Input names:', [i.name for i in sess.get_inputs()])
print('Output names:', [o.name for o in sess.get_outputs()])

# Run with example data
result = sess.run(None, {
    sess.get_inputs()[0].name: np.full((1, 250), 440.0, dtype=np.float32),
    sess.get_inputs()[1].name: np.full((1, 250), -30.0, dtype=np.float32),
    sess.get_inputs()[2].name: np.array([0], dtype=np.int32),
})

for i, out in enumerate(sess.get_outputs()):
    print(f'{out.name}: shape={result[i].shape}, min={result[i].min():.4f}, max={result[i].max():.4f}')

print('\nValidation passed — model produces non-zero synthesis parameters.')

In [ ]:
# 9. Download the ONNX file
from google.colab import files
files.download(OUTPUT_PATH)

print(f'\nDone! Upload to HuggingFace with:')
print(f'  hf upload jcosta33/vocoder-models ddsp_decoder.onnx ddsp-decoder/ddsp_decoder.onnx --repo-type model')
print(f'\nThen update DDSP_MODEL_SIZE_BYTES in ddspInstrumentCatalog.ts to: {file_size}')